# GAFIME Quickstart Tutorial 🚀

Welcome to GAFIME (GPU-Accelerated Feature Interaction Mining Engine)! 
This interactive notebook will guide you through the process of generating highly optimized feature interactions on your GPU or CPU.

Let's get started by importing the library and generating some dummy feature data!

In [ ]:
import numpy as np

np.random.seed(42)
n_samples = 100_000
n_features = 50

# Generate random dense feature data (e.g., outputs from a previous deep learning model)
X = np.random.randn(n_samples, n_features).astype(np.float32)

# Generate a random binary classification target
y = np.random.randint(0, 2, size=n_samples).astype(np.float32)

print(f"Created dataset with {n_samples:,} samples and {n_features} features.")

## Initializing the Engine

GAFIME automatically selects the fastest available backend (CUDA > Metal > CPU OpenMP > Rust > Python Baseline).
Let's initialize the engine and see which backend it picks up!

In [ ]:
from gafime.engine import FeatureInteractionEngine

engine = FeatureInteractionEngine(backend='auto', verbose=True)
print(f"Active Backend: {engine.backend.__class__.__name__}")

## Evaluating Feature Interactions

We have 50 features, meaning there are `50 * 49 / 2 = 1,225` unique pairwise combinations. 
GAFIME evaluates how "good" the interaction of two features (e.g., $f_i \times f_j$) is relative to the target using high-speed Information Value (IV) or Pearson correlation metrics.

Let's evaluate all 1,225 pairs instantly!

In [ ]:
import time

# We'll use the 'multiply' operator and evaluate using the default 'pearson' metric.
start = time.perf_counter()
scores = engine.evaluate_interactions(X, y, metric='pearson', operator='multiply')
elapsed = time.perf_counter() - start

print(f"Evaluated pairwise interactions in {elapsed*1000:.2f} ms!")
print(f"Returned scores array shape: {scores.shape}")

## Finding the Best Interactions
Now that we've dumped the raw interaction metric scores, let's extract the Top N highest-performing pairs to feed into our final downstream model (such as XGBoost, CatBoost, or a Neural Network).

In [ ]:
from gafime.planning.combinations import find_top_k_interactions

top_pairs = find_top_k_interactions(scores, k=5)

print("Top 5 most powerful feature combinations:")
for rank, (score, feat_i, feat_j) in enumerate(top_pairs, 1):
    print(f"#{rank} | Interaction (Feature {feat_i} x Feature {feat_j}): Score = {score:.4f}")